In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
from sklearn.metrics import r2_score, mean_squared_error
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
df_train = pd.read_csv("/kaggle/input/house-prices-advanced-regression-techniques/train.csv")
df_test = pd.read_csv("/kaggle/input/house-prices-advanced-regression-techniques/test.csv")

# Explore Numerical Data

In [ ]:
num_df = df_train.select_dtypes(include=[np.number]).drop('Id', axis=1)
corr_matrix = num_df.corr()['SalePrice'].abs().sort_values(ascending=False)

corr_matrix.head(15)

In [ ]:
plt.figure(figsize=(10, 8))
sns.heatmap(num_df.corr())
plt.title("Correlation Heatmap")

In [ ]:
plt.figure(figsize=(8, 5))
sns.scatterplot(x='GrLivArea', y='SalePrice', data=df_train)
plt.title('Price vs Living Area - Outliers Far from Trend')
plt.show()

In [ ]:
outlier_mask = df_train['GrLivArea'] < 4000

y = np.log(df_train['SalePrice'])
df_train_clean = df_train[outlier_mask].copy()
y_clean = y[outlier_mask]

dropped = df_train[~outlier_mask][['GrLivArea', 'SalePrice', 'OverallQual', 'Id']]
print("Dropped Outliers:\n", dropped)

top_cat = ['Neighborhood', 'ExterQual', 'KitchenQual', 'GarageType', 'SaleCondition']
top_num = ['OverallQual', 'GrLivArea', 'GarageCars', 'GarageArea', 'TotalBsmtSF', 
           '1stFlrSF', 'FullBath', 'TotRmsAbvGrd']
num_features = top_num
cat_features = top_cat
X_full = df_train_clean[num_features + cat_features]
X_test_full = df_test[num_features + cat_features]

# Straight to the pipeline

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import cross_val_score
from sklearn.metrics import make_scorer, mean_squared_error
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline


X_num = df_train_clean[top_num].fillna(0)

pipe = Pipeline([('scaler', StandardScaler()), ('lr', LinearRegression())])

rmse_scorer = make_scorer(mean_squared_error, squared=False)
cv_rmse = cross_val_score(pipe, X_num, y_clean, cv=5, scoring='neg_root_mean_squared_error')
print(f"Phase 1 CV RMSE: {-cv_rmse.mean():.4f} (+/- {cv_rmse.std():.4f})")


In [ ]:
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer




num_imputer = SimpleImputer(strategy='median')
cat_imputer = SimpleImputer(strategy='most_frequent')
cat_encoder = OneHotEncoder(drop='first', handle_unknown='ignore', sparse_output=False)

preprocessor = ColumnTransformer([
    ('num', Pipeline([('impute', num_imputer), ('scale', StandardScaler())]), num_features),
    ('cat', Pipeline([('impute', cat_imputer), ('encode', cat_encoder)]), cat_features)
])

pipe2 = Pipeline([('preproc', preprocessor), ('lr', LinearRegression())])


rmse_scorer = make_scorer(mean_squared_error, squared=False)
cv_rmse2 = cross_val_score(pipe2, X_full, y_clean, cv=5, scoring='neg_root_mean_squared_error')
print(f"Phase 2 CV RMSE: {-cv_rmse2.mean():.4f} (+/- {cv_rmse2.std():.4f})")

In [ ]:
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor, StackingRegressor
from sklearn.model_selection import cross_val_score

ridge = Ridge(alpha=10)

estimators = [
    ('lr', LinearRegression()),
    ('rf', RandomForestRegressor(n_estimators=100, random_state=42, max_depth=10)) 
]
stack = StackingRegressor(estimators=estimators, final_estimator=ridge, cv=5)


pipe3 = Pipeline([('preproc', preprocessor), ('stack', stack)])  # preprocessor from Phase 2

rmse_scorer = make_scorer(mean_squared_error, squared=False)
cv_rmse3 = cross_val_score(pipe3, X_full, y_clean, cv=5, scoring='neg_root_mean_squared_error')
print(f"Phase 3 CV RMSE: {-cv_rmse3.mean():.4f} (+/- {cv_rmse3.std():.4f})")

In [ ]:
from sklearn.model_selection import KFold, cross_val_score
from sklearn.feature_selection import VarianceThreshold
from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler, PowerTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import RidgeCV
from sklearn.metrics import make_scorer, mean_squared_log_error

# Load
train = df_train
test  = df_test
y = train["SalePrice"].astype(float)
X = train.drop(columns=["SalePrice", "Id"], errors="ignore")
X_test = test.drop(columns=["Id"], errors="ignore")

# Remove classic outliers
bad = (train["GrLivArea"] > 4000) & (train["SalePrice"] < 300000)
X, y = X.loc[~bad].copy(), y.loc[~bad].copy()

# Columns (unchanged)
num_cols = X.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = X.select_dtypes(exclude=[np.number]).columns.tolist()

# Preprocessing: dense output + no Yeo-Johnson
num_pipe = Pipeline([
    ("impute", SimpleImputer(strategy="median")),
    ("vt", VarianceThreshold(threshold=0.0)),   # drop constant cols (prevents transformer issues)
    ("scale", StandardScaler()),
])
cat_pipe = Pipeline([
    ("impute", SimpleImputer(strategy="most_frequent")),
    ("ohe", OneHotEncoder(handle_unknown="ignore")),  # leave default sparse; we force dense below
])
pre = ColumnTransformer([
    ("num", num_pipe, num_cols),
    ("cat", cat_pipe, cat_cols),
], remainder="drop", sparse_threshold=0.0)  # force dense to avoid sparse_cg

# Model: RidgeCV on log-target; dense => uses svd/cholesky, not sparse_cg
ridge = RidgeCV(
    alphas=np.logspace(-3, 3, 50),
    cv=KFold(n_splits=10, shuffle=True, random_state=42),
    scoring="neg_mean_squared_error",
)

model = Pipeline([
    ("pre", pre),
    ("ttr", TransformedTargetRegressor(regressor=ridge, func=np.log1p, inverse_func=np.expm1)),
])

# RMSLE scorer (matches Kaggle metric)
rmsle_scorer = make_scorer(
    lambda yt, yp: np.sqrt(mean_squared_log_error(np.clip(yt, 0, None), np.clip(yp, 0, None))),
    greater_is_better=False
)

# CV
cv = KFold(n_splits=10, shuffle=True, random_state=42)
scores = -cross_val_score(model, X, y, scoring=rmsle_scorer, cv=cv, n_jobs=-1)
(scores.mean(), scores.std())

In [ ]:

model.fit(X, y)
pred = model.predict(X_test)
pred = np.clip(pred, 0, None)

submission = pd.DataFrame({
    'Id': df_test['Id'],
    'SalePrice': pred
})
submission.to_csv('submission.csv', index=False)
print("Submission saved")